# CodeGen Capstone — Checkpoint 3
**Full RAG pipeline: FAISS + AST + hybrid retrieval, Top-K experiments, Claude Sonnet 4 comparison**

Builds a 2000+ sample code index, compares dense/AST/hybrid retrieval across K in {1,3,5,10}
plus dynamic Top-K, and produces the small-LM vs. LLM-no-RAG vs. LLM+RAG vs. fine-tuned+RAG
comparison. Set `ANTHROPIC_API_KEY` (Colab secret or env var) before running Section 4.

In [1]:
REPO_URL = "https://github.com/<your-org>/codegen-rag-capstone.git"  # only used as a fallback; ignored if the project is already on Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/codegen-rag-capstone"
LOCAL_CLONE_DIR = "/content/codegen-rag-capstone"

import os

if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "src")):
    PROJECT_DIR = DRIVE_PROJECT_DIR
    print("Found project on Google Drive:", PROJECT_DIR)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "src")):
        PROJECT_DIR = DRIVE_PROJECT_DIR
        print("Found project on Google Drive:", PROJECT_DIR)
    elif "<your-org>" not in REPO_URL:
        os.system(f"git clone --depth 1 {REPO_URL} {LOCAL_CLONE_DIR}")
        PROJECT_DIR = LOCAL_CLONE_DIR
    else:
        raise FileNotFoundError(
            f"Could not find the project at {DRIVE_PROJECT_DIR} on Google Drive, and REPO_URL is "
            "still the placeholder. Upload the codegen-rag-capstone/ folder to 'My Drive' (so it "
            "lives at exactly that path), or set REPO_URL to your pushed GitHub repo."
        )

os.chdir(PROJECT_DIR)

import sys

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

from codegen_rag.utils.env_setup import bootstrap_environment

settings = bootstrap_environment(install_deps=True, mount_drive=True)
print("Project root:", settings.root_dir)

Mounted at /content/drive
Found project on Google Drive: /content/drive/MyDrive/codegen-rag-capstone
2026-07-20 20:12:03 | INFO     | codegen_rag.utils.env_setup | Installing dependencies from /content/drive/MyDrive/codegen-rag-capstone/requirements.txt ...
2026-07-20 20:12:44 | INFO     | codegen_rag.utils.env_setup | Removing preinstalled torchao (incompatible with current peft on Colab)
2026-07-20 20:12:45 | INFO     | codegen_rag.utils.env_setup | Google Drive mounted. Project root: /content/drive/MyDrive/CodeGen_Capstone
2026-07-20 20:12:45 | INFO     | codegen_rag.utils.env_setup | Folder structure ready under /content/drive/MyDrive/CodeGen_Capstone
2026-07-20 20:12:51 | INFO     | codegen_rag.utils.env_setup | Environment ready | Colab=True | GPU=True (Tesla T4, 14.6 GB) | CUDA=12.8 | root=/content/drive/MyDrive/CodeGen_Capstone
Project root: /content/drive/MyDrive/CodeGen_Capstone


## 1. Build the RAG corpus index (2000+ code samples)
Reuses the CodeParrot subset from Checkpoint 1 config plus the CoDocBench splits, embedded
with `codegen-350M-multi`'s own hidden states (`CodeGenModel.embed`).

In [2]:
from codegen_rag.data.downloaders import download_codeparrot_subset
from codegen_rag.data.preprocessors import parse_codocbench_directory

codeparrot_cfg = settings.data["codeparrot"]
codeparrot_samples = download_codeparrot_subset(
    settings.resolve_path(codeparrot_cfg["clone_dir"]),
    languages=codeparrot_cfg["languages"],
    max_samples=max(codeparrot_cfg["max_samples"], 2000),
    hf_dataset=codeparrot_cfg["hf_dataset"],
)

codoc_cfg = settings.data["codocbench"]
codoc_records = [
    r.to_dict()
    for r in parse_codocbench_directory(settings.resolve_path(codoc_cfg["clone_dir"]), languages=["python"])
]

corpus = [{"code": s["code"], "intent": s.get("path", "")} for s in codeparrot_samples if s["language"] == "Python"]
corpus += [{"code": r["code"], "intent": r.get("intent", r.get("function_name", ""))} for r in codoc_records]
print(f"RAG corpus size: {len(corpus)} samples (target: 2000+)")

2026-07-20 20:12:52 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
2026-07-20 20:12:55 | INFO     | datasets | PyTorch version 2.11.0+cu128 available.
2026-07-20 20:12:55 | INFO     | datasets | Polars version 1.35.2 available.
2026-07-20 20:12:55 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-07-20 20:12:55 | INFO     | datasets | JAX version 0.7.2 available.
2026-07-20 20:12:57 | INFO     | codegen_rag.data.downloaders | CodeParrot subset already cached at /content/drive/MyDrive/CodeGen_Capstone/data/raw/codeparrot
2026-07-20 20:13:04 | INFO     | codegen_rag.data.preprocessors | Parsed 18248 raw CoDocBench records across ['python']
2026-07-20 20:13:05 | INFO     | codegen_rag.data.preprocessors | Deduplicated 18248 -> 9089 records
RAG corpus size: 15755 samples (target: 2000+)


In [3]:
from codegen_rag.models.codegen_wrapper import load_model_for_task
from codegen_rag.rag.corpus_indexing import build_indexes_from_corpus, save_indexes

base_model = load_model_for_task(settings)
embed_dim = settings.model["embeddings"]["embedding_dim"]

def embed_texts(texts):
    return base_model.embed(texts, normalize=settings.model["embeddings"]["normalize"]).numpy()

dense_index, ast_index = build_indexes_from_corpus(corpus, embed_fn=embed_texts, embedding_dim=embed_dim)
save_indexes(dense_index, ast_index, settings.path_for("faiss_index"))
print(f"Dense index: {len(dense_index)} vectors (IVF used: {dense_index.used_ivf})")
print(f"AST index: {len(ast_index)} entries")

2026-07-20 20:13:23 | INFO     | codegen_rag.models.codegen_wrapper | Loading base model Salesforce/codegen-350M-multi on cuda (dtype=torch.float16)
2026-07-20 20:13:24 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-20 20:13:25 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 20:13:25 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"
2026-07-20 20:13:25 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

2026-07-20 20:13:25 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


2026-07-20 20:13:25 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

2026-07-20 20:13:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 20:13:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/tokenizer.json "HTTP/1.1 200 OK"
2026-07-20 20:13:40 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Salesforce/codegen-350M-multi/b25de779e2044ed5e7707505dea0e5a9bb08556a/tokenizer.json "HTTP/1.1 200 OK"


tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

2026-07-20 20:13:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Salesforce/codegen-350M-multi/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-20 20:13:44 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 32/15755 corpus entries
2026-07-20 20:13:58 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 352/15755 corpus entries
2026-07-20 20:14:13 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 672/15755 corpus entries
2026-07-20 20:14:28 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 992/15755 corpus entries
2026-07-20 20:14:43 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 1312/15755 corpus entries
2026-07-20 20:14:59 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 1632/15755 corpus entries
2026-07-20 20:15:14 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 1952/15755 corpus entries
2026-07-20 20:15:29 | INFO     | codegen_rag.rag.corpus_indexing | Embedded 2272/15755 corpus entries
2026-07-20 20:15:4

<unknown>:75: SyntaxWarning: invalid escape sequence '\ '
<unknown>:148: SyntaxWarning: invalid escape sequence '\.'
<unknown>:149: SyntaxWarning: invalid escape sequence '\.'
<unknown>:150: SyntaxWarning: invalid escape sequence '\.'
<unknown>:96: SyntaxWarning: invalid escape sequence '\%'
<unknown>:5: SyntaxWarning: invalid escape sequence '\/'
<unknown>:5: SyntaxWarning: invalid escape sequence '\/'
<unknown>:35: SyntaxWarning: invalid escape sequence '\/'
<unknown>:16: SyntaxWarning: invalid escape sequence '\d'
<unknown>:42: SyntaxWarning: invalid escape sequence '\@'
<unknown>:43: SyntaxWarning: invalid escape sequence '\#'
<unknown>:44: SyntaxWarning: invalid escape sequence '\?'
<unknown>:45: SyntaxWarning: invalid escape sequence '\%'
<unknown>:46: SyntaxWarning: invalid escape sequence '\/'
<unknown>:47: SyntaxWarning: invalid escape sequence '\+'
<unknown>:48: SyntaxWarning: invalid escape sequence '\-'
<unknown>:49: SyntaxWarning: invalid escape sequence '\_'
<unknown>:50:

2026-07-20 20:26:06 | INFO     | codegen_rag.rag.ast_retrieval | Built AST retrieval index over 15755 entries
2026-07-20 20:26:06 | INFO     | codegen_rag.rag.corpus_indexing | Built dense (15755 vectors) and AST (15755 entries) indexes
2026-07-20 20:26:15 | INFO     | codegen_rag.rag.faiss_index | Saved FAISS index (15755 vectors) to /content/drive/MyDrive/CodeGen_Capstone/faiss_index/dense
2026-07-20 20:26:17 | INFO     | codegen_rag.rag.corpus_indexing | Saved RAG indexes to /content/drive/MyDrive/CodeGen_Capstone/faiss_index
Dense index: 15755 vectors (IVF used: True)
AST index: 15755 entries


## 2. Top-K experiments (dense / AST / hybrid x K in {1,3,5,10} + dynamic)

In [6]:
from codegen_rag.data.preprocessors import train_val_test_split
from codegen_rag.models.generation_config import GenerationConfig
from codegen_rag.rag.topk_experiment import run_topk_experiment, select_best_configuration

splits = train_val_test_split(codoc_records, seed=settings.project.seed)
topk_eval_records = [r for r in splits["test"] if r.get("intent")][:30]

gen_cfg = GenerationConfig(**settings.model["generation"]["program_synthesis"])
small_lm_generate_fn = lambda prompt: base_model.generate(prompt, gen_cfg)[0]

topk_df = run_topk_experiment(
    topk_eval_records,
    dense_index,
    ast_index,
    embed_fn=lambda q: embed_texts([q])[0],
    generate_fn=small_lm_generate_fn,
    query_key="intent",
    reference_key="code",
)
topk_df

2026-07-20 20:30:56 | INFO     | codegen_rag.rag.topk_experiment | strategy=dense K=1 codebleu=0.0811
2026-07-20 20:35:09 | INFO     | codegen_rag.rag.topk_experiment | strategy=dense K=3 codebleu=0.0764
2026-07-20 20:39:18 | INFO     | codegen_rag.rag.topk_experiment | strategy=dense K=5 codebleu=0.0789
2026-07-20 20:43:30 | INFO     | codegen_rag.rag.topk_experiment | strategy=dense K=10 codebleu=0.0931
2026-07-20 20:47:38 | INFO     | codegen_rag.rag.topk_experiment | strategy=dense K=dynamic codebleu=0.0786
2026-07-20 20:52:02 | INFO     | codegen_rag.rag.topk_experiment | strategy=ast K=1 codebleu=0.0989
2026-07-20 20:56:13 | INFO     | codegen_rag.rag.topk_experiment | strategy=ast K=3 codebleu=0.0826
2026-07-20 21:00:35 | INFO     | codegen_rag.rag.topk_experiment | strategy=ast K=5 codebleu=0.0957
2026-07-20 21:04:26 | INFO     | codegen_rag.rag.topk_experiment | strategy=ast K=10 codebleu=0.0797
2026-07-20 21:08:25 | INFO     | codegen_rag.rag.topk_experiment | strategy=ast K=

,strategy,top_k,dynamic,n_examples,codebleu
0,dense,1,False,30,0.081118
1,dense,3,False,30,0.076441
2,dense,5,False,30,0.078866
3,dense,10,False,30,0.093126
4,dense,dynamic,True,30,0.078614
5,ast,1,False,30,0.098860
6,ast,3,False,30,0.082620
7,ast,5,False,30,0.095740
8,ast,10,False,30,0.079678
9,ast,dynamic,True,30,0.080309


In [7]:
best_config = select_best_configuration(topk_df)
print("Best retrieval configuration:", best_config)

Best retrieval configuration: {'strategy': 'ast', 'top_k': 1, 'dynamic': False, 'n_examples': 30, 'codebleu': 0.09885992168403097}


## 3. Fine-tuned model inside RAG
Reuses the Rust LoRA adapter from Checkpoint 1/2 — swap the adapter path if you want to
evaluate a different checkpoint.

In [9]:
from codegen_rag.models.codegen_wrapper import CodeGenModel
from codegen_rag.training.checkpoint_manager import CheckpointManager

rust_ckpt_mgr = CheckpointManager(settings.path_for("checkpoints") / "rust_full")
best_rust_ckpt = rust_ckpt_mgr.find_resume_point("best")

fine_tuned_model = CodeGenModel(
    model_name=settings.base_model.name,
    adapter_path=best_rust_ckpt.checkpoint_dir if best_rust_ckpt else None,
)
fine_tuned_generate_fn = lambda prompt: fine_tuned_model.generate(prompt, gen_cfg)[0]

2026-07-20 21:31:22 | INFO     | codegen_rag.training.checkpoint_manager | Resuming from best checkpoint: step=2200
2026-07-20 21:31:23 | INFO     | codegen_rag.models.codegen_wrapper | Loading full fine-tuned checkpoint from /content/drive/MyDrive/CodeGen_Capstone/checkpoints/rust_full/checkpoint-002200 (no adapter_config.json -> not a LoRA adapter, loading complete weights directly)


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

## 4. Claude Sonnet 4 comparison + 4-tier table

In [11]:
from codegen_rag.models.model_registry import UpperBoundLLMClient
from codegen_rag.rag.pipeline import RAGPipeline
from codegen_rag.rag.topk_experiment import run_four_tier_comparison

llm_client = UpperBoundLLMClient(
    primary=settings.upper_bound_llm.primary,
    fallback_order=settings.upper_bound_llm.fallback_order,
    max_tokens=settings.upper_bound_llm.max_tokens,
    temperature=settings.upper_bound_llm.temperature,
)
llm_generate_fn = lambda prompt: llm_client.generate(prompt).text

best_strategy = best_config["strategy"]
best_k = best_config["top_k"] if best_config["top_k"] != "dynamic" else 5

rag_pipeline = RAGPipeline(
    dense_index=dense_index, ast_index=ast_index, embed_fn=lambda q: embed_texts([q])[0],
    strategy=best_strategy, top_k=int(best_k),
)
fine_tuned_rag_pipeline = RAGPipeline(
    dense_index=dense_index, ast_index=ast_index, embed_fn=lambda q: embed_texts([q])[0],
    strategy=best_strategy, top_k=int(best_k),
)

four_tier_df = run_four_tier_comparison(
    topk_eval_records[:15],  # LLM API calls cost money — keep this bounded
    small_lm_generate_fn=small_lm_generate_fn,
    llm_generate_fn=llm_generate_fn,
    rag_pipeline=rag_pipeline,
    fine_tuned_rag_pipeline=fine_tuned_rag_pipeline,
    query_key="intent",
    reference_key="code",
)
four_tier_df

2026-07-20 21:32:10 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:32:20 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:32:46 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:32:56 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:33:15 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:33:25 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:33:56 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:34:06 | INFO     | httpx | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-20 21:34:39 | INFO     | httpx | HTTP Request: POST https://api.

,model_tier,n_examples,codebleu
0,small_lm_baseline,15,0.072563
1,llm_no_rag,15,0.249486
2,llm_rag,15,0.175515
3,fine_tuned_rag,15,0.075151


## 5. Charts and report

In [12]:
from codegen_rag.evaluation.visualizations import (
    generate_markdown_report,
    plot_bar_comparison,
    plot_topk_sweep,
)

results_dir = settings.path_for("results")
four_tier_chart = plot_bar_comparison(
    four_tier_df, "model_tier", "codebleu", results_dir / "four_tier_comparison.png",
    title="Small LM vs LLM vs LLM+RAG vs Fine-tuned+RAG",
)
topk_chart = plot_topk_sweep(topk_df, results_dir / "topk_sweep.png")

report_path = generate_markdown_report(
    {
        "Best retrieval configuration": str(best_config),
        "Top-K sweep results": topk_df,
        "4-tier comparison": four_tier_df,
        "Charts": [four_tier_chart, topk_chart],
    },
    results_dir / "checkpoint3_report.md",
)
print("Report written to", report_path)

2026-07-20 21:41:21 | INFO     | codegen_rag.evaluation.visualizations | Saved chart to /content/drive/MyDrive/CodeGen_Capstone/results/four_tier_comparison.png
2026-07-20 21:41:22 | INFO     | codegen_rag.evaluation.visualizations | Saved top-K sweep chart to /content/drive/MyDrive/CodeGen_Capstone/results/topk_sweep.png
2026-07-20 21:41:22 | INFO     | codegen_rag.evaluation.visualizations | Wrote markdown report to /content/drive/MyDrive/CodeGen_Capstone/results/checkpoint3_report.md
Report written to /content/drive/MyDrive/CodeGen_Capstone/results/checkpoint3_report.md


## Checkpoint 3 completion checklist
- [x] CodeBERT/codegen embeddings for retrieval
- [x] FAISS vector database (2000+ samples target)
- [x] Token (dense) retrieval
- [x] AST retrieval
- [x] Hybrid retrieval (RRF fusion)
- [x] Top-K experiments (K = 1, 3, 5, 10)
- [x] Dynamic Top-K experiments
- [x] Context packing / prompt augmentation
- [x] Claude Sonnet 4 integration (with GPT-5 fallback)
- [x] Fine-tuned model inside RAG
- [x] Base vs Fine-Tuned vs LLM vs LLM+RAG comparison
- [x] Charts, graphs, markdown report

Proceed to `04_checkpoint4_deployment.ipynb` next (or run `scripts/serve_api.py` / the Streamlit app directly).